# Why SMOTENC + ENN instead of SMOTE + ENN?

This presentation notebook compares the same raw 80% training split with two resampling methods:

1. **SMOTENC + ENN** — the saved result from the project's prior preprocessing run.
2. **SMOTE + ENN** — a comparator run in memory with the same settings.

The 20% raw holdout is excluded throughout. No comparison CSVs are saved; only W&B figures and tables are logged.

## 1. Load the exact raw training partition

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import wandb
import yaml
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import EditedNearestNeighbours
from scipy.stats import ks_2samp, wasserstein_distance
from sklearn.model_selection import train_test_split

REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / 'configs').is_dir():
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / 'configs').is_dir():
    raise FileNotFoundError('Run this notebook from inside the repository.')
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.data.load import load_raw, split_features_target

sns.set_theme(style='whitegrid', context='notebook')
cfg = yaml.safe_load((REPO_ROOT / 'configs' / 'resample_smoteenn.yaml').read_text())
data_cfg = cfg['data']
sampler_cfg = cfg['smoteenn']
TARGET = data_cfg['target_column']
SEED = int(cfg['random_seed'])
TEST_SIZE = float(data_cfg['test_size'])
CATEGORICAL_FEATURES = sampler_cfg['categorical_features']

raw_df = load_raw(REPO_ROOT / data_cfg['input_path'], target_column=TARGET)
raw_train_df, _ = train_test_split(
    raw_df, test_size=TEST_SIZE, random_state=SEED, stratify=raw_df[TARGET]
)
raw_train_df = raw_train_df.reset_index(drop=True)
X_raw, y_raw = split_features_target(raw_train_df, TARGET)
FEATURES = X_raw.columns.tolist()

print(f'Raw training split: {len(raw_train_df):,} rows ({1 - TEST_SIZE:.0%} of raw data)')
print(f'Raw holdout excluded: {len(raw_df) - len(raw_train_df):,} rows ({TEST_SIZE:.0%})')
print(f'Categorical features protected by SMOTENC: {len(CATEGORICAL_FEATURES)}')

## 2. Reuse the saved SMOTENC + ENN result and run the SMOTE + ENN comparator

The saved SMOTENC + ENN dataset was generated by the project's preprocessing pipeline. Only the SMOTE + ENN comparator runs here; both methods use the same random seed, sampling strategy, neighbour count, and ENN cleaning settings.

In [ ]:
def clean_with_enn(X, y):
    enn = EditedNearestNeighbours(
        sampling_strategy=sampler_cfg['enn'].get('sampling_strategy', 'all'),
        n_neighbors=sampler_cfg['enn'].get('n_neighbors', 3),
        kind_sel=sampler_cfg['enn'].get('kind_sel', 'all'),
        n_jobs=sampler_cfg.get('n_jobs', -1),
    )
    return enn.fit_resample(X, y)

def resample_then_clean(oversampler):
    X_over, y_over = oversampler.fit_resample(X_raw, y_raw)
    X_clean, y_clean = clean_with_enn(X_over, y_over)
    X_clean = pd.DataFrame(X_clean, columns=FEATURES).reset_index(drop=True)
    y_clean = pd.Series(y_clean, name=TARGET).astype(int).reset_index(drop=True)
    return pd.concat([y_clean, X_clean], axis=1)

common_smote_kwargs = dict(
    sampling_strategy=sampler_cfg.get('sampling_strategy', 'auto'),
    random_state=SEED,
    k_neighbors=sampler_cfg['smote'].get('k_neighbors', 5),
)

smotenc_output_path = REPO_ROOT / data_cfg['output_path']
if not smotenc_output_path.is_file():
    raise FileNotFoundError(
        f'Saved SMOTENC + ENN output not found: {smotenc_output_path}. '
        'Run scripts/run_resample.py first.'
    )
smotenc_enn_df = pd.read_csv(smotenc_output_path)
expected_columns = [TARGET, *FEATURES]
if set(smotenc_enn_df.columns) != set(expected_columns):
    raise ValueError('Saved SMOTENC + ENN output has a different schema from the raw training data.')
smotenc_enn_df = smotenc_enn_df[expected_columns]

smote_enn_df = resample_then_clean(SMOTE(**common_smote_kwargs))

print(f'Saved SMOTENC + ENN: {smotenc_enn_df.shape[0]:,} rows from {smotenc_output_path.name}')
print(f'SMOTE + ENN:   {smote_enn_df.shape[0]:,} rows')

## 3. Start a W&B comparison run

In [ ]:
WANDB_MODE = cfg.get('wandb', {}).get('mode', 'online')
run = wandb.init(
    project=cfg.get('wandb', {}).get('project', 'diabetes-brfss-ml'),
    entity=cfg.get('wandb', {}).get('entity'),
    name='mahdi-smotenc-enn-vs-smote-enn',
    job_type='resampling-comparison',
    tags=['mahdi', 'smotenc-enn', 'smote-enn', 'data-audit'],
    mode=WANDB_MODE,
    config={
        'random_seed': SEED,
        'test_size': TEST_SIZE,
        'categorical_feature_count': len(CATEGORICAL_FEATURES),
        'raw_train_rows': len(raw_train_df),
        'smotenc_enn_rows': len(smotenc_enn_df),
        'smote_enn_rows': len(smote_enn_df),
    },
)
print(f'W&B run: {run.url if run.url else WANDB_MODE}')

## 4. Compact all-feature distribution comparison

Each panel overlays normalized distributions for the raw training data and both pipelines. Every feature uses shared bins across its three datasets.

In [ ]:
DATASETS = {
    'Raw 80% training': raw_train_df,
    'SMOTENC + ENN': smotenc_enn_df,
    'SMOTE + ENN': smote_enn_df,
}

COLORS = {
    'Raw 80% training': '#4C78A8',
    'SMOTENC + ENN': '#54A24B',
    'SMOTE + ENN': '#E45756'
}

PRESENTATION_FEATURES = [
    'genhlth',
    'education',
    'income',
    'highchol',
    'highbp'
]

# 5 rows = features
# 3 columns = techniques
fig, axes = plt.subplots(
    nrows=len(PRESENTATION_FEATURES),
    ncols=len(DATASETS),
    figsize=(15, 18),
    sharey='row'
)

# Column titles
for col_idx, dataset_name in enumerate(DATASETS.keys()):
    axes[0, col_idx].set_title(
        dataset_name,
        fontsize=12,
        fontweight='bold',
        pad=12
    )

for row_idx, feature in enumerate(PRESENTATION_FEATURES):

    # Get all values for this feature so every technique
    # uses the same bins
    all_values = [
        df[feature].dropna().to_numpy()
        for df in DATASETS.values()
    ]

    combined = np.concatenate(all_values)

    raw_unique = np.unique(all_values[0])

    if len(raw_unique) <= 20:
        lo = combined.min()
        hi = combined.max()

        bins = np.linspace(
            lo - 0.5,
            hi + 0.5,
            max(
                8,
                min(
                    41,
                    int((hi - lo) * 3) + 2
                )
            )
        )
    else:
        bins = np.histogram_bin_edges(
            combined,
            bins=30
        )

    # Plot each technique in its own cell
    for col_idx, (dataset_name, df) in enumerate(DATASETS.items()):

        ax = axes[row_idx, col_idx]

        values = df[feature].dropna().to_numpy()

        # Filled bars
        ax.hist(
            values,
            bins=bins,
            weights=np.full(
                len(values),
                1 / len(values)
            ),
            color=COLORS[dataset_name],
            alpha=0.75,
            edgecolor='black',
            linewidth=0.5
        )

        # Feature name on left
        if col_idx == 0:
            ax.set_ylabel(
                f'{feature}\nshare',
                fontsize=10,
                fontweight='bold'
            )

        ax.set_xlabel(
            'Feature value',
            fontsize=9
        )

        ax.tick_params(
            labelsize=8
        )

        ax.grid(
            axis='y',
            alpha=0.2
        )

# Overall title
fig.suptitle(
    'Selected Feature Distributions: Raw vs. SMOTENC + ENN vs. SMOTE + ENN',
    fontsize=16,
    fontweight='bold',
    y=0.995
)

# Legend
handles = [
    plt.Rectangle(
        (0, 0),
        1,
        1,
        facecolor=COLORS[name],
        edgecolor='black',
        alpha=0.75,
        label=name
    )
    for name in DATASETS.keys()
]

fig.legend(
    handles=handles,
    labels=list(DATASETS.keys()),
    loc='lower center',
    ncol=3,
    frameon=False,
    bbox_to_anchor=(0.5, 0.005)
)

fig.tight_layout(
    rect=(0, 0.04, 1, 0.98)
)

run.log({
    'comparison/selected_feature_distributions': wandb.Image(fig)
})

plt.show()
plt.close(fig)

## 5. Categorical-value integrity: the decision figure

A categorical feature is invalid when its value is not one of the valid raw categories. This figure isolates the evidence for choosing SMOTENC: it selects valid categories, while ordinary SMOTE interpolates between category codes.

In [ ]:
integrity_rows = []

for feature in CATEGORICAL_FEATURES:
    valid_values = set(raw_train_df[feature].unique())

    for name, frame in DATASETS.items():
        invalid_share = float(
            (~frame[feature].isin(valid_values)).mean()
        )

        fractional_share = float(
            (~np.isclose(
                frame[feature],
                np.round(frame[feature])
            )).mean()
        )

        integrity_rows.append({
            'feature': feature,
            'dataset': name,
            'invalid_category_share': invalid_share,
            'fractional_value_share': fractional_share,
        })

integrity_df = pd.DataFrame(integrity_rows)


# ---------------------------------------------------------
# SMOTE invalid-category results
# ---------------------------------------------------------

smote_invalid = (
    integrity_df
    .query("dataset == 'SMOTE + ENN'")
    .sort_values(
        'invalid_category_share',
        ascending=False
    )
    .reset_index(drop=True)
)

smotenc_max_invalid = (
    integrity_df
    .query("dataset == 'SMOTENC + ENN'")[
        'invalid_category_share'
    ]
    .max()
)


# ---------------------------------------------------------
# Plot
# ---------------------------------------------------------

fig, ax = plt.subplots(figsize=(16, 8))

x = np.arange(len(smote_invalid))
values = smote_invalid['invalid_category_share'].to_numpy()

bars = ax.bar(
    x,
    values,
    width=0.7,
    color='#E45756',
    alpha=0.9,
    edgecolor='black',
    linewidth=0.5
)


# ---------------------------------------------------------
# Value labels above each bar
# ---------------------------------------------------------

max_value = values.max()

# Give enough space above bars for labels
if max_value > 0:
    ax.set_ylim(0, max_value * 1.22)
else:
    ax.set_ylim(0, 1)

y_range = ax.get_ylim()[1]

for bar, value in zip(bars, values):

    ax.text(
        bar.get_x() + bar.get_width() / 2,
        value + y_range * 0.015,
        f'{value:.1%}',
        ha='center',
        va='bottom',
        fontsize=9,
        fontweight='bold'
    )


# ---------------------------------------------------------
# SMOTENC = 0% reference line
# ---------------------------------------------------------

ax.axhline(
    0,
    color='#54A24B',
    linewidth=5,
    label='SMOTENC + ENN: 0% invalid across all 20 features'
)


# ---------------------------------------------------------
# Axis labels / ticks
# ---------------------------------------------------------

ax.set_xticks(x)

ax.set_xticklabels(
    smote_invalid['feature'],
    rotation=45,
    ha='right',
    fontsize=9
)

ax.set_ylabel(
    'Share of resampled rows with an invalid category',
    fontsize=11
)

ax.set_xlabel(
    'Categorical feature',
    fontsize=11,
    labelpad=10
)


# ---------------------------------------------------------
# Title
# ---------------------------------------------------------

ax.set_title(
    'Ordinary SMOTE Creates Invalid Categorical Values',
    fontsize=16,
    fontweight='bold',
    pad=22
)


# ---------------------------------------------------------
# Subtitle / explanation
# ---------------------------------------------------------

ax.text(
    0.5,
    1.015,
    f'SMOTENC + ENN maximum invalid share: '
    f'{smotenc_max_invalid:.0%}  |  '
    f'SMOTE + ENN interpolates categorical codes',
    transform=ax.transAxes,
    ha='center',
    va='bottom',
    fontsize=10,
    fontweight='bold',
    color='#386641'
)


# ---------------------------------------------------------
# Grid
# ---------------------------------------------------------

ax.grid(
    axis='y',
    alpha=0.2
)

ax.set_axisbelow(True)


# ---------------------------------------------------------
# Legend
# ---------------------------------------------------------

ax.legend(
    loc='upper right',
    frameon=False,
    fontsize=9
)


# Remove unnecessary top/right borders
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)


# Extra space for rotated labels and title
fig.subplots_adjust(
    top=0.84,
    bottom=0.22,
    left=0.08,
    right=0.98
)


# ---------------------------------------------------------
# W&B logging
# ---------------------------------------------------------

run.log({
    'comparison/categorical_integrity': wandb.Image(fig),
    'comparison/categorical_integrity_table': wandb.Table(
        dataframe=integrity_df
    ),
})

plt.show()
plt.close(fig)

## 6. Balance and distribution shift: the supporting evidence

Both pipelines address class imbalance. The heatmap shows how much each feature's distribution moves away from the raw training split, using the Kolmogorov-Smirnov statistic (larger values mean a larger shift).

In [ ]:
class_rows = []
for name, frame in DATASETS.items():
    for label, share in frame[TARGET].value_counts(normalize=True).sort_index().items():
        class_rows.append({'dataset': name, 'class': int(label), 'share': share})
class_df = pd.DataFrame(class_rows)

shift_rows = []
for feature in FEATURES:
    before = raw_train_df[feature].to_numpy()
    for pipeline_name, frame in [('SMOTENC + ENN', smotenc_enn_df), ('SMOTE + ENN', smote_enn_df)]:
        after = frame[feature].to_numpy()
        shift_rows.append({
            'feature': feature, 'pipeline': pipeline_name,
            'ks_statistic': ks_2samp(before, after).statistic,
            'wasserstein_distance': wasserstein_distance(before, after),
        })
shift_df = pd.DataFrame(shift_rows).sort_values(['feature', 'pipeline']).reset_index(drop=True)
shift_heatmap = (
    shift_df.pivot(index='feature', columns='pipeline', values='ks_statistic')
    .sort_values('SMOTE + ENN', ascending=False)
)

fig, (ax_balance, ax_shift) = plt.subplots(
    1, 2, figsize=(15, 8), gridspec_kw={'width_ratios': [1, 2.1]}
)
sns.barplot(
    data=class_df, x='class', y='share', hue='dataset', hue_order=list(DATASETS),
    palette=COLORS, ax=ax_balance,
)
ax_balance.set(title='Class balance', xlabel='Diabetes class', ylabel='Share of rows', ylim=(0, 1))
ax_balance.legend(title='', fontsize=8, loc='upper right')
ax_balance.grid(axis='y', alpha=0.2)

sns.heatmap(
    shift_heatmap, annot=True, fmt='.3f', cmap='YlOrRd', linewidths=0.5,
    cbar_kws={'label': 'KS statistic'}, ax=ax_shift,
)
ax_shift.set(title='Feature distribution shift from raw training data', xlabel='', ylabel='')
fig.suptitle('Both methods rebalance classes; SMOTENC preserves category validity', fontsize=15, fontweight='bold', y=1.02)
fig.tight_layout()
run.log({
    'comparison/balance_and_distribution_shift': wandb.Image(fig),
    'comparison/class_balance_table': wandb.Table(dataframe=class_df),
    'comparison/distribution_shift_table': wandb.Table(dataframe=shift_df),
})
plt.show()
plt.close(fig)

## 7. Finish the W&B run

In [ ]:
run.finish()
print('Comparison figures and tables logged to W&B.')